## 1. Verify the Python environment

checks which Python environment the notebook uses, prints
the TensorFlow version, and checks whether EfficientNetB0 is available.
It does not create or train a model.

In [1]:
import sys
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("EfficientNetB0 available:", callable(tf.keras.applications.EfficientNetB0))

Python: c:\Users\SewwandiWijesinghe\SE4050-Deep-Learning-Assignment\venv\Scripts\python.exe
TensorFlow: 2.21.0
EfficientNetB0 available: True


## 2. Check dataset folders and class counts

locates the data folder and counts images in each class.
The dataset contains 5,600 Training images and 1,600 Testing images.
Each class has 1,400 Training images and 400 Testing images.

In [2]:
from pathlib import Path

# Find the data folder from either the project or notebooks folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data"
print("Dataset location:", data_dir)

classes = ["glioma", "meningioma", "notumor", "pituitary"]

for split in ["Training", "Testing"]:
    print(f"\n{split}:")
    for class_name in classes:
        folder = data_dir / split / class_name
        if not folder.is_dir():
            print(f"  {class_name}: FOLDER NOT FOUND")
        else:
            count = sum(
                1 for file in folder.iterdir()
                if file.is_file()
                and file.suffix.lower() in {".jpg", ".jpeg", ".png"}
            )
            print(f"  {class_name}: {count} images")

Dataset location: c:\Users\SewwandiWijesinghe\SE4050-Deep-Learning-Assignment\data

Training:
  glioma: 1400 images
  meningioma: 1400 images
  notumor: 1400 images
  pituitary: 1400 images

Testing:
  glioma: 400 images
  meningioma: 400 images
  notumor: 400 images
  pituitary: 400 images


## 3. Check image readability

This cell attempts to open and decode every image using Pillow.
Files that cannot be read are recorded for inspection.
All 7,200 images passed this check. No original files were modified.

In [3]:
from PIL import Image

image_files = sorted(
    file
    for split in ["Training", "Testing"]
    for file in (data_dir / split).rglob("*")
    if file.is_file()
    and file.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

unreadable_images = []

for file in image_files:
    try:
        with Image.open(file) as img:
            img.convert("RGB").load()
    except Exception as error:
        unreadable_images.append((str(file), str(error)))

print("Images checked:", len(image_files))
print("Unreadable images:", len(unreadable_images))

for file, error in unreadable_images[:10]:
    print(file, error)

Images checked: 7200
Unreadable images: 0


## 4. Check for exact duplicate files

In [4]:
import hashlib
from collections import defaultdict

files_by_hash = defaultdict(list)

for file in image_files:
    fingerprint = hashlib.sha256(file.read_bytes()).hexdigest()
    files_by_hash[fingerprint].append(file)

duplicate_groups = [
    files for files in files_by_hash.values()
    if len(files) > 1
]

train_test_overlap = [
    files for files in duplicate_groups
    if {file.relative_to(data_dir).parts[0] for file in files}
    == {"Training", "Testing"}
]

print("Groups of identical files:", len(duplicate_groups))
print("Groups appearing in both Training and Testing:",
      len(train_test_overlap))

for group in train_test_overlap[:5]:
    print("\nMatching files:")
    for file in group:
        print(" ", file.relative_to(data_dir))

Groups of identical files: 153
Groups appearing in both Training and Testing: 0


The check found 153 duplicate groups and no exact duplicates shared
between the Training and Testing folders.

This check does not detect every visually similar image or prove
that the folders contain scans from different patients.

## 5. Inspect duplicate locations and labels

This cell counts duplicate groups and extra copies separately within
Training and Testing. It also checks whether identical files have
different class labels.

These findings will guide consistent duplicate handling across the team.


In [5]:
# Count duplicate groups and extra copies in each folder.
for split in ["Training", "Testing"]:
    groups = [
        group for group in duplicate_groups
        if all(
            file.relative_to(data_dir).parts[0] == split
            for file in group
        )
    ]

    # Keep one copy per group when calculating extra copies.
    extra_copies = sum(len(group) - 1 for group in groups)

    print(f"\n{split}:")
    print("  Duplicate groups:", len(groups))
    print("  Extra copies:", extra_copies)

# Check whether identical files appear under different class labels.
conflicting_groups = [
    group for group in duplicate_groups
    if len({
        file.relative_to(data_dir).parts[1]
        for file in group
    }) > 1
]

print(
    "\nIdentical-image groups with different class labels:",
    len(conflicting_groups)
)


Training:
  Duplicate groups: 138
  Extra copies: 171

Testing:
  Duplicate groups: 15
  Extra copies: 16

Identical-image groups with different class labels: 0


## Dataset audit findings

- All 7,200 images were readable.
- Training contains 138 exact-duplicate groups and 171 extra copies.
- Testing contains 15 exact-duplicate groups and 16 extra copies.
- No exact file duplicates were found across Training and Testing.
- No exact-duplicate groups had conflicting class labels.
- Original files have not been modified.
- Shared duplicate handling and train/validation assignments must be
  settled before the final model comparison.
- These checks do not establish patient-level independence.